# Sprite → 3D avec Pixal3D (Colab)
Génère un GLB texturé depuis ton sprite détouré, sur le GPU gratuit de Colab.

**Avant de lancer** : *Exécution → Modifier le type d'exécution → GPU* (T4 suffit en mode économe en mémoire).
Cellules à exécuter dans l'ordre. L'installation (cellule 2) compile des extensions CUDA : compte 20 à 40 min, et il faut la refaire à chaque nouvelle session Colab.

⚠️ Notebook non testé sur GPU : si une cellule échoue, copie-moi le message d'erreur.

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],capture_output=True,text=True).stdout)
assert torch.cuda.is_available(), "Active un GPU : Exécution > Modifier le type d'exécution"
cap = torch.cuda.get_device_capability()
print("torch", torch.__version__, "capacité GPU", cap)

In [ ]:
%%bash
set -e
cd /content
[ -d Pixal3D ] || git clone https://github.com/TencentARC/Pixal3D.git --recursive
[ -d TRELLIS.2 ] || git clone -b main https://github.com/microsoft/TRELLIS.2.git --recursive
# Base TRELLIS.2 (sans flash-attn : la T4 ne le gère pas, on utilisera SDPA)
cd TRELLIS.2
. ./setup.sh --basic --nvdiffrast --nvdiffrec --cumesh --o-voxel --flexgemm
cd ../Pixal3D
pip install -r requirements.txt
pip install https://github.com/LDYang694/Storages/releases/download/20260430/utils3d-0.0.2-py3-none-any.whl

In [ ]:
import os, torch
arch = "%d.%d" % torch.cuda.get_device_capability()
os.environ["NATTEN_CUDA_ARCH"] = arch
os.environ["NATTEN_N_WORKERS"] = "2"
!cd /content/Pixal3D && pip install natten==0.21.0 --no-build-isolation

In [ ]:
# Facultatif : certains poids (DINOv3, matting) peuvent demander un compte Hugging Face.
# Ajoute un secret Colab nommé HF_TOKEN (icône clé à gauche) puis exécute :
try:
    from google.colab import userdata
    from huggingface_hub import login
    login(userdata.get("HF_TOKEN"))
    print("connecté à Hugging Face")
except Exception as e:
    print("pas de token HF (ok si les poids sont publics) :", e)

In [ ]:
# Ton sprite : upload, sinon Zimut.png du dépôt SpriteBuilder
import os, urllib.request
from google.colab import files
os.makedirs("/content/entree", exist_ok=True)
print("Choisis ton PNG détouré (annule pour prendre Zimut.png du dépôt)")
try:
    up = files.upload()
    chemin = "/content/entree/" + next(iter(up))
    open(chemin, "wb").write(next(iter(up.values())))
except Exception:
    chemin = "/content/entree/Zimut.png"
    urllib.request.urlretrieve("https://raw.githubusercontent.com/CharlieG42/SpriteBuilder/main/sprites_3d/entrees/Zimut.png", chemin)
print("image :", chemin)

In [ ]:
%%bash -s "$chemin"
cd /content/Pixal3D
export ATTN_BACKEND=sdpa
export PYTHONPATH=/content/TRELLIS.2:$PYTHONPATH
python inference.py --image "$1" --output /content/sortie.glb --low_vram --resolution 1024
ls -la /content/sortie.glb

In [ ]:
from google.colab import files
files.download("/content/sortie.glb")